# 03 - Train All Models (multi-seed, identical protocol)
**Addresses comments 1, 3, 5.**  Every model is trained with *exactly the same* split manifest, augmentation, optimiser (AdamW + cosine), epochs, patience and checkpoint rule, for every seed.

How to use this notebook (read once):
1. **Cell 2** loads the REAL data (it stops with an error if the data are not found - there is no synthetic fallback) and lets you choose the models, seeds and epochs.
2. **Cell 3** measures the speed of your machine on a few batches and prints an estimate of the total time **before** you start.
3. **Cell 4** trains. It prints a line every epoch, and a progress heartbeat during the first epoch. It is **safe to interrupt** (Kernel -> Interrupt): finished runs are kept and skipped next time; only the run in progress restarts.
4. **Cell 5** shows the table of finished runs and the mean +/- std per model.

For multi-hour jobs prefer the terminal: `python scripts/run_experiments.py --mode pilot --configs baseline,film_control,full --seeds 0,1`

In [ ]:
import os, sys, json, warnings
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import config as C
C.banner()
import time
from utils.experiment import run_one, CONFIGS, load_results, build_model, compute_loss
from utils.data import make_loaders
splits, man = C.get_splits()          # raises FileNotFoundError if the KLA data are missing - fix SEMICON_DATA, do not continue
print({k: len(v) for k, v in splits.items()}, "| split sha1:", man["sha1"][:12])
assert len(splits["train"]) > 500 or C.MODE == "smoke", "Too few training images - this is not the KLA data set; check SEMICON_DATA."

# ------------------------------- EDIT THIS BLOCK ---------------------------------
GROUPS = {
    "core":     ["baseline", "global_gc", "film_control", "ldmh_loss_only", "full"],
    "ablation": ["full_no_penalty", "full_clamp", "full_no_bal", "full_no_div"],
    "external": ["NAFNet-small", "Restormer-small", "SwinIR-small", "NAFNet-base", "Restormer-base", "SwinIR-base"],
}
MODELS = ["baseline", "film_control", "full"]        # pilot default: the three that decide gate 2.  Use GROUPS["core"] for all five.
SEEDS = list(C.SEEDS)                                # e.g. [0, 1]
EPOCHS = C.EPOCHS                                    # e.g. 10 for a quick CPU check (state it in the paper if you report it)
# ---------------------------------------------------------------------------------
plan = [(n, s) for n in MODELS for s in SEEDS]
print(f"{len(plan)} runs planned: models={MODELS} seeds={SEEDS} epochs={EPOCHS} device={C.device()}")

MODE=pilot  epochs=30  seeds=[0, 1]  device=cpu  out=d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow\results_pilot
{'train': 2560, 'val': 320, 'test': 320} | split sha1: 31d7258b8c36
6 runs planned: models=['baseline', 'film_control', 'full'] seeds=[0, 1] epochs=30 device=cpu


In [ ]:
# Speed test: time 10 training batches for each chosen model, then project the cost. Nothing is saved.
import torch
dev = C.device()
tl_, _, _ = make_loaders(splits, C.BATCH_SIZE, seed=0, cache=C.CACHE)
n_batches = len(tl_); speeds = {}
for name in MODELS:
    cfg = dict(CONFIGS[name]); torch.manual_seed(0); model = build_model(cfg).to(dev).train()
    opt = torch.optim.AdamW(model.parameters(), lr=C.LR); it = iter(tl_); times = []
    for b in range(min(11, n_batches)):
        noisy, gt, gt_ds, _ = next(it); noisy, gt, gt_ds = noisy.to(dev), gt.to(dev), gt_ds.to(dev)
        t = time.time(); opt.zero_grad(); loss, _ = compute_loss(cfg, model(noisy), gt, gt_ds, noisy); loss.backward(); opt.step()
        if b > 0: times.append(time.time() - t)              # skip the first (warm-up) batch
    speeds[name] = (np.mean(times) if times else 0.0) * n_batches / 60
    print(f"{name:14s} ~{speeds[name]:.1f} min / epoch  ({(np.mean(times) if times else 0.0):.2f} s per batch, {n_batches} batches)")
total_h = sum(speeds[n] * EPOCHS * len(SEEDS) for n in MODELS) / 60
print(f"\nESTIMATED TOTAL TRAINING TIME: ~{total_h:.1f} hours  (validation adds a few %; early stopping may shorten runs)")
if total_h > 24: print("-> More than a day. Consider fewer EPOCHS, fewer MODELS/SEEDS, or a GPU (see README).")

baseline       ~58.4 min / epoch  (10.95 s per batch, 320 batches)
film_control   ~61.6 min / epoch  (11.55 s per batch, 320 batches)
full           ~63.8 min / epoch  (11.97 s per batch, 320 batches)

ESTIMATED TOTAL TRAINING TIME: ~183.8 hours  (validation adds a few %; early stopping may shorten runs)
-> More than a day. Consider fewer EPOCHS, fewer MODELS/SEEDS, or a GPU (see README).


In [ ]:
t0 = time.time(); done = []
try:
    for k, (name, seed) in enumerate(plan):
        r = run_one(name, seed, splits, run_root=C.RUNS, epochs=EPOCHS, batch_size=C.BATCH_SIZE, lr=C.LR, wd=C.WEIGHT_DECAY,
                    clip=C.GRAD_CLIP, patience=C.PATIENCE, device=C.device(), cache=C.CACHE, verbose=True)
        done.append(r)
        print(f"==> [{k+1}/{len(plan)}] {name} seed{seed} DONE | best_ep={r['best_epoch']} val_psnr={r['val']['psnr']:.3f} "
              f"test_psnr={r['test']['psnr']:.3f} | {r['train_minutes']:.1f} min | total elapsed {(time.time()-t0)/60:.1f} min\n", flush=True)
except KeyboardInterrupt:
    print("\nInterrupted. Finished runs are saved; run this cell again to continue (the interrupted run restarts).")

In [ ]:
res = load_results(C.RUNS)
cols = ["config", "seed", "params", "best_epoch", "val_psnr", "test_psnr", "test_ssim", "test_hp_err", "minutes"]
print(res[cols].round(4).to_string(index=False))
print("\nmean +/- std over seeds (descriptive only; the paired statistics are in notebook 04):")
g = res.groupby("config")[["test_psnr", "test_ssim", "test_hp_err"]].agg(["mean", "std", "count"]).round(4)
print(g.to_string())
res.to_csv(os.path.join(C.TABLES, "all_runs_summary.csv"), index=False)